# UpLift — Notebook 02: Preprocessing & Class Balancing
**Project:** UpLift — Predictive Maintenance for Transit Accessibility Equipment  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What This Notebook Does

1. Loads the dataset from Notebook 01
2. Encodes categorical variables
3. Splits data into **training** (80%) and **holdout** (20%) sets — holdout is locked and never used for tuning
4. Applies **SMOTE** (Synthetic Minority Over-sampling Technique) to the training set only
5. Saves processed arrays for model training in Notebook 03

### Why SMOTE?

Only ~25% of equipment units fail in any given 30-day window. Without intervention, a naive model learns to predict 'no failure' for everything and achieves 75% accuracy while being useless. SMOTE generates synthetic examples of the minority class (failures) so the model sees equal representation during training. Think of it like a fire drill — you run them precisely *because* real fires are rare.

In [ ]:
# Install dependencies (run once in Google Colab)
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_csv('uplift_equipment_data.csv')
print(f'Loaded: {df.shape[0]} records, {df.shape[1]} columns')
print(f'\nClass balance (raw):')
print(df['outage_within_30_days'].value_counts())
df.head(3)

## 2. Encode Categorical Features

In [ ]:
df_model = df.drop(columns=['equipment_id'])

# Label encode categoricals
cat_cols = ['equipment_type', 'agency', 'manufacturer']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

joblib.dump(encoders, 'uplift_encoders.pkl')
print('\nEncoders saved: uplift_encoders.pkl')

## 3. Train / Holdout Split

In [ ]:
FEATURE_COLS = [
    'equipment_type', 'equipment_age_years', 'manufacturer',
    'days_since_last_maintenance', 'unplanned_outages_12mo',
    'total_usage_load_millions', 'days_since_parts_replacement',
    'inspection_score', 'avg_monthly_temp_f', 'avg_daily_ridership',
    'floors_served', 'is_transfer_hub', 'agency'
]
TARGET = 'outage_within_30_days'

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

# Stratified split preserves class ratio in both sets
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]} records')
print(f'Holdout set:   {X_holdout.shape[0]} records  ← LOCKED. No tuning on this.')
print(f'\nTraining class balance:')
print(y_train.value_counts())

## 4. Apply SMOTE to Training Set Only

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('After SMOTE:')
print(f'Training set size: {X_train.shape[0]} → {X_train_resampled.shape[0]}')
print(f'\nNew class balance:')
print(pd.Series(y_train_resampled).value_counts())

In [ ]:
# Visualize class balance before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('UpLift — Class Balance Before and After SMOTE', fontweight='bold', fontsize=13)

labels = ['No Outage (0)', 'Outage (1)']
before = y_train.value_counts().sort_index()
after = pd.Series(y_train_resampled).value_counts().sort_index()

axes[0].bar(labels, before.values, color=['#4CAF50', '#F44336'], alpha=0.85, edgecolor='white')
axes[0].set_title('Before SMOTE', fontweight='bold')
axes[0].set_ylabel('Record Count')
for i, v in enumerate(before.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

axes[1].bar(labels, after.values, color=['#4CAF50', '#F44336'], alpha=0.85, edgecolor='white')
axes[1].set_title('After SMOTE (Training Set Only)', fontweight='bold')
axes[1].set_ylabel('Record Count')
for i, v in enumerate(after.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('uplift_smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_smote_balance.png')

## 5. Save Processed Data

In [ ]:
import numpy as np
np.save('X_train.npy', X_train_resampled)
np.save('y_train.npy', y_train_resampled)
np.save('X_holdout.npy', X_holdout.values)
np.save('y_holdout.npy', y_holdout.values)
joblib.dump(FEATURE_COLS, 'uplift_feature_cols.pkl')

print('Saved:')
print('  X_train.npy      ← SMOTE-balanced training features')
print('  y_train.npy      ← SMOTE-balanced training labels')
print('  X_holdout.npy    ← Holdout features (untouched)')
print('  y_holdout.npy    ← Holdout labels (untouched)')
print('  uplift_feature_cols.pkl')
print(f'\n→ Notebook 03: Model Training')

## Design Notes

| Decision | Rationale |
|---|---|
| **SMOTE on training only** | Applying SMOTE to holdout data would contaminate the evaluation — we need holdout to reflect real-world class distribution |
| **Stratified split** | Ensures both sets have ~25% positive class; prevents lucky/unlucky splits |
| **No scaling applied** | XGBoost and tree-based models are scale-invariant; scaling not required |
| **Label encoding** | Ordinal encoding for categoricals; XGBoost handles these natively |

➡️ **Next:** Notebook 03 — Model Training, Threshold Selection & Evaluation